# GR-Lite: Fashion Image Retrieval Tutorial

This notebook demonstrates how to load and use the [GR-Lite](https://huggingface.co/srpone/gr-lite) model for fashion image retrieval.

GR-Lite is a ViT-L/16 model (303M params) fine-tuned from DINOv3 for fashion retrieval. It produces 1024-dim L2-normalized image embeddings.

**What you'll learn:**
1. Load GR-Lite from HuggingFace Hub
2. Extract image embeddings
3. Compute similarity between images
4. Run image retrieval on LookBench

## 1. Installation

In [ ]:
!pip install -q transformers torch torchvision pillow

## 2. Load the Model

GR-Lite is hosted on HuggingFace and can be loaded with `AutoModel`. The `trust_remote_code=True` flag is required because the model uses a custom architecture.

In [ ]:
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained("srpone/gr-lite", trust_remote_code=True)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Model loaded on {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 3. Preprocessing

GR-Lite expects 336x336 RGB images normalized with ImageNet statistics.

In [ ]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((336, 336)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## 4. Extract Embeddings

The model returns `pooler_output` (L2-normalized CLS embedding) and `last_hidden_state` (full patch sequence).

In [ ]:
from PIL import Image
import requests
from io import BytesIO

# Load a sample image (replace with your own image path)
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/model_doc/vit_architecture.jpg"
try:
    response = requests.get(url, timeout=5)
    image = Image.open(BytesIO(response.content)).convert("RGB")
except Exception:
    # Fallback: create a dummy image for demonstration
    image = Image.new("RGB", (400, 400), color=(128, 128, 128))

# Preprocess and extract embedding
pixel_values = transform(image).unsqueeze(0).to(device)  # [1, 3, 336, 336]

with torch.no_grad():
    output = model(pixel_values)

embedding = output.pooler_output  # [1, 1024]
print(f"Embedding shape: {embedding.shape}")
print(f"L2 norm: {embedding.norm().item():.4f}")
print(f"First 5 values: {embedding[0, :5].tolist()}")

## 5. Image Similarity

Since embeddings are L2-normalized, cosine similarity is just a dot product.

In [ ]:
# Create a second image (slightly modified) for comparison
image2 = image.rotate(15)

pixel_values2 = transform(image2).unsqueeze(0).to(device)

with torch.no_grad():
    output2 = model(pixel_values2)

emb1 = output.pooler_output   # from the original image
emb2 = output2.pooler_output  # from the rotated image

# Cosine similarity (dot product since embeddings are L2-normalized)
similarity = (emb1 @ emb2.T).item()
print(f"Cosine similarity (original vs rotated): {similarity:.4f}")

## 6. Batch Feature Extraction

For retrieval workloads, use a DataLoader for efficient batch processing.

In [ ]:
from torch.utils.data import DataLoader, Dataset

class ImageListDataset(Dataset):
    """Simple dataset wrapping a list of PIL images."""
    def __init__(self, images, transform):
        self.images = images
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.transform(self.images[idx])

# Create some dummy images for demonstration
dummy_images = [Image.new("RGB", (400, 400), color=(r, g, b))
                for r, g, b in [(255, 0, 0), (0, 255, 0), (0, 0, 255), (128, 128, 0)]]

dataset = ImageListDataset(dummy_images, transform)
loader = DataLoader(dataset, batch_size=2, shuffle=False)

all_embeddings = []
with torch.no_grad():
    for batch in loader:
        out = model(batch.to(device))
        all_embeddings.append(out.pooler_output.cpu())

embeddings = torch.cat(all_embeddings, dim=0)  # [4, 1024]
print(f"Batch embeddings shape: {embeddings.shape}")

# Pairwise similarity matrix
sim_matrix = embeddings @ embeddings.T
print(f"\nPairwise similarity matrix:\n{sim_matrix.numpy().round(3)}")

## 7. Image Retrieval Example

Given a query image, find the most similar images from a gallery by ranking cosine similarities.

In [ ]:
def retrieve(query_emb, gallery_embs, top_k=5):
    """Retrieve top-K most similar gallery items for a query."""
    # query_emb: [1, D], gallery_embs: [N, D]
    scores = (query_emb @ gallery_embs.T).squeeze(0)  # [N]
    top_k_indices = scores.argsort(descending=True)[:top_k]
    return top_k_indices, scores[top_k_indices]

# Use first image as query, rest as gallery
query_emb = embeddings[0:1]      # [1, 1024]
gallery_embs = embeddings[1:]    # [3, 1024]

indices, scores = retrieve(query_emb, gallery_embs, top_k=3)

print("Query: Image 0 (red)")
print("\nRetrieval results:")
colors = ["green", "blue", "yellow"]
for rank, (idx, score) in enumerate(zip(indices, scores)):
    print(f"  Rank {rank+1}: Image {idx.item()+1} ({colors[idx]}) — similarity: {score.item():.4f}")

## Resources

- [GR-Lite on HuggingFace](https://huggingface.co/srpone/gr-lite)
- [LookBench Paper](https://arxiv.org/abs/2601.14706)
- [LookBench Dataset](https://huggingface.co/datasets/srpone/look-bench)
- [LookBench GitHub](https://github.com/SerendipityOneInc/look-bench)